# Chapter 8 — Planning, Decomposition and Replanning

*A plan is a typed list of steps, not a paragraph.*

## Objective

**Why.** Sometimes an agent should think ahead rather than interleave thought and action. A plan lets the rest of the system inspect what the agent intends — the budget counts the steps, the audit log records the commitment, the evaluator compares planned vs actual.

**What.** A plan is a *typed list of steps*, not a paragraph. That typing is what makes it inspectable — and validatable before it runs.

**How.** Plan the same task three ways (fixed workflow, LM-generated, graph search), decompose into typed subtasks, trigger a replan on failure — then validate a whole plan against the workflow before any tool fires.

In [ ]:
from glassloop.core import TaskSpec
from glassloop.planning import (
    GraphSearchPlanner, LMPlanner, Plan, PlanStep,
    ReplanReason, ReplanTrigger, WorkflowPlanner,
    decompose, replan, should_replan,
)

## A reference task

We will plan to handle a customer complaint: classify, retrieve policy, draft a response.

In [ ]:
task = TaskSpec(
    goal='Handle a customer complaint about overdraft fees',
    inputs={'message': 'I was charged $35 unfairly', 'start': 'start'},
    expected_outputs=['drafted'],
)

## WorkflowPlanner — predefined steps

Transparent, reliable, inflexible. Use this when the procedure is settled.

In [ ]:
workflow = WorkflowPlanner([
    PlanStep(id='classify', description='classify the complaint', action_hint='classify'),
    PlanStep(id='retrieve', description='retrieve relevant policy', action_hint='search'),
    PlanStep(id='draft',    description='draft a response',         action_hint='compose'),
])
for s in workflow.plan(task).steps:
    print(f'  {s.id}: {s.description}')

## LMPlanner — a JSON plan from a real model

When the procedure varies by task, a model can produce the plan. `LMPlanner` prompts a real small model — the library's `QwenAdapter` (a local Qwen2.5-3B-Instruct) — for a JSON array and parses it into typed steps. A reply that isn't valid JSON is rejected, not guessed at.

In [ ]:
from glassloop.models import QwenAdapter

lm_plan = LMPlanner(QwenAdapter()).plan(task)
for s in lm_plan.steps:
    print(f'  {s.id}: {s.description[:60]}')

# The planner parses the reply as JSON and refuses anything else rather than
# guessing — a non-JSON reply raises, reaching the audit log instead of becoming
# a silent bad plan:
import json
try:
    json.loads('Sure! First classify the complaint, then draft a reply.')
except json.JSONDecodeError:
    print('non-JSON reply rejected: LMPlanner raises ValueError')

## GraphSearchPlanner — BFS over an action graph

Rigorous on small domains. Costly to scale. Useful when you have an explicit state machine.

In [ ]:
graph = {
    'start':      {'classify': 'classified'},
    'classified': {'search':   'searched'},
    'searched':   {'compose':  'drafted'},
}
graph_plan = GraphSearchPlanner(graph).plan(task)
for s in graph_plan.steps:
    print(f'  {s.id}: {s.action_hint}')

## Decomposition: one task becomes many

Each plan step becomes a typed subtask that inherits constraints and adds a parent reference.

In [ ]:
subtasks = decompose(task, workflow.plan(task))
for sub in subtasks:
    print(f'  {sub.goal!r}  inputs={list(sub.inputs.keys())}')

## Replanning fires on tool failure

`should_replan(last_tool_result)` returns a typed `ReplanTrigger` when the last tool call failed. `replan(planner, task, trigger)` annotates the task with the trigger so the new plan can be different.

In [ ]:
trigger = should_replan({'success': False, 'error': 'rate limit'})
print(trigger)
new_plan = replan(workflow, task, trigger)
print('new plan length:', len(new_plan))

## An optional check: validating the plan before it runs

A plan is a *hypothesis*; a wrong one is cheapest to catch before any tool fires. The per-step gate (Ch6) checks each call as it happens; the **plan-level invariant** checks the whole plan one moment earlier. Each `(step, has_enables, next_step)` transition has a geodesic distance against the workflow graph, and the plan's mean distance — its **cumulative coherence drift** — must stay under a calibrated budget. A plan that skips or reorders steps drifts past the budget and is refused, with no tool run and no state changed. **Optional; for safety-critical workflows.**

In [ ]:
import torch
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore
from glassloop.gms_backend import GMSPlanGate

_root = Path('.') if Path('data/gms_banking_store').exists() else Path('..')
store = GMSExpertStore(
    DocGMSConfig(store_path=str(_root / 'data' / 'gms_banking_store')),
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
)
store.load()
gate = GMSPlanGate(store, drift_budget=1.35)   # budget calibrated in App. C

for label, plan in [
    ('drafting path',   ['classify', 'extract', 'search_policy', 'draft_response']),
    ('escalation path', ['classify', 'extract', 'flag_regulatory', 'escalate']),
    ('skips two steps', ['classify', 'draft_response']),
]:
    v = gate.validate(plan)
    print(f'  {label:<16} drift={v.drift:.3f}  admissible={v.admissible}')

## Anti-patterns flagged here

- Letting the LM generate plans in free text.
- Plans without validation rules.
- Replanning loops without a budget gate.

In [ ]:
# Self-check
assert len(workflow.plan(task)) == 3
assert len(lm_plan) >= 1          # a real model's plan length varies
assert len(graph_plan) == 3
assert trigger.reason == ReplanReason.TOOL_FAILURE
print('OK')

## Exercises

Worked solutions follow — plan generation → decomposition → validation. Exercise 8.2 loads Qwen2.5-3B-Instruct (~30s).

In [ ]:
# Exercise 8.1 — from plan to work list
plan = workflow.plan(task)
for sub in decompose(task, plan):
    print(f'  goal={sub.goal!r}')
    print(f'    parent_goal={sub.inputs.get("_parent_goal")!r}  step_id={sub.inputs.get("_step_id")!r}')
assert all('_parent_goal' in s.inputs and '_step_id' in s.inputs for s in decompose(task, plan))

In [ ]:
# Exercise 8.2 — replan on failure, but bound it
attempts, MAX_REPLANS = 0, 3
result = {'success': False, 'error': 'rate limit'}   # a tool that keeps failing
while attempts < MAX_REPLANS:
    trig = should_replan(result)
    if trig is None:
        break
    replan(workflow, task, trig)
    attempts += 1
print(f'replanned {attempts} times, then stopped (would escalate instead)')
print('should_replan on a success:', should_replan({'success': True}))

In [ ]:
# Exercise 8.3 — reject a bad plan before it runs
plans = {
    'drafting':   ['classify', 'extract', 'search_policy', 'draft_response'],
    'escalation': ['classify', 'extract', 'flag_regulatory', 'escalate'],
    'skip':       ['classify', 'draft_response'],
}
verdicts = {name: gate.validate(p) for name, p in plans.items()}
for name, v in verdicts.items():
    print(f'  {name:<11} drift={v.drift:.3f}  admissible={v.admissible}')
print('rejected at plan-time:', [n for n, v in verdicts.items() if not v.admissible])
# One geodesic score over the whole plan, before any tool runs: the skip plan is
# refused with no state change. At execution-time the illegal step would surface
# only after the earlier steps had already acted on the world.